<p style="font-family: 'Courier New', Courier, monospace; font-size: 40px; font-weight: bold; color: blue;  text-align: center;">
  LoRaWAN Path Loss Measurements in an Indoor Setting: DATASET ANIMATION
</p>

###  Notebook Summary
This Notebook animates the last two weeks of the cleaned dataset (`cleaned_dataset_per_device.csv`, written by notebook 05): temperature, humidity, CO₂, PM2.5, pressure, RSSI and SNR of the six end devices, one frame every 30 minutes, saved as `two_weeks_all_sensors.gif` in `all_data_files/`. It stands alone: notebooks 01–06 do not need it. Rendering takes about seven minutes.

In [1]:
# Libraries/Packages Used:
import os                                  # For creating the output folder.
import pandas as pd                        # For data manipulation and analysis.
import numpy as np                         # For numerical computations and handling arrays.
import matplotlib.pyplot as plt            # For creating static visualizations.
import matplotlib.dates as mdates          # For handling dates in matplotlib.
from matplotlib.animation import FuncAnimation, PillowWriter  # For saving animations as GIFs.
%matplotlib inline

In [2]:
# Load the cleaned dataset (notebook 05)
cleaned_df = pd.read_csv('../all_data_files/cleaned_dataset_per_device.csv')
print(f"{len(cleaned_df):,} rows from {cleaned_df['time'].min()} to {cleaned_df['time'].max()}")

2,660,274 rows from 2024-10-01 00:01:07.420593+00:00 to 2025-09-30 23:59:55.971870+00:00


<p style="font-family: 'Courier New', Courier, monospace; font-size: 30px; font-weight: bold; color: blue;  text-align: left;">
  Plot and save Animation for some dataframe variables over time
</p>

In [3]:
def make_two_weeks_all_sensors_animation(
    cleaned_df,
    outfile="../all_data_files/two_weeks_all_sensors.gif",
    days_back=14,
    fps=10,
    frame_step="1H",   # one frame per hour over the two weeks
):
    """
    Create a two-weeks-long animation with all devices and 7 parameters:
    temperature, humidity, co2, pm25, pressure, rssi, snr.

    The animation gradually reveals data from start to end of the two weeks
    (cumulative), which looks like a streaming timeline.
    """

    # Ensure folder exists
    os.makedirs(os.path.dirname(outfile), exist_ok=True)

    # Prepare data
    df = cleaned_df.copy()
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time")

    # Keep only last `days_back` days
    t_end = df["time"].max()
    t_start = t_end - pd.Timedelta(days=days_back)
    df = df[(df["time"] >= t_start) & (df["time"] <= t_end)]

    if df.empty:
        print("No data in the requested time window. Adjust days_back.")
        return

    # Parameters and devices
    params = ["temperature", "humidity", "co2", "pm25", "pressure", "rssi", "snr"]
    # Filter to params that actually exist in the dataframe
    params = [p for p in params if p in df.columns]

    if not params:
        print("None of the expected parameters are in cleaned_df.")
        return

    device_ids = df["device_id"].unique()
    n_params = len(params)

    # Build time grid for frames
    frame_times = pd.date_range(start=t_start, end=t_end, freq=frame_step)
    n_frames = len(frame_times)
    if n_frames == 0:
        print("No frames to render. Check frame_step and data range.")
        return

    print(f"Animating last {days_back} days, {len(device_ids)} devices, "
          f"{n_params} parameters, {n_frames} frames...")

    # Figure and axes
    fig_height = 2.2 * n_params
    fig, axes = plt.subplots(n_params, 1, sharex=True, figsize=(11, fig_height))
    if n_params == 1:
        axes = [axes]

    # Color map for devices
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(device_ids), 1)))

    # One line per (param, device)
    lines = {param: {} for param in params}
    for ax, param in zip(axes, params):
        for dev, col in zip(device_ids, colors):
            (line,) = ax.plot([], [], lw=1.2, label=str(dev), color=col)
            lines[param][dev] = line

        ax.set_ylabel(param)
        ax.grid(True, alpha=0.3)

    axes[0].set_title(
        f"Sensor readings over last {days_back} days "
        f"({len(device_ids)} devices, {n_params} parameters)"
    )
    axes[-1].set_xlabel("Time")
    axes[0].legend(loc="upper right", fontsize=8, ncol=2)

    # Time axis formatting
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d-%b\n%H:%M"))

    def init():
        for param in params:
            for dev in device_ids:
                lines[param][dev].set_data([], [])
        return [line for d in lines.values() for line in d.values()]

    def update(frame_idx):
        t_frame = frame_times[frame_idx]
        df_frame = df[df["time"] <= t_frame]

        for ax, param in zip(axes, params):
            # Update each device line for this parameter
            for dev in device_ids:
                d = df_frame[df_frame["device_id"] == dev]
                x = d["time"].values
                y = d[param].values
                lines[param][dev].set_data(x, y)

            # Fixed x range over the full month
            ax.set_xlim(t_start, t_end)

            # Adaptive y range per parameter
            series = df_frame[param]
            if len(series) > 0:
                ymin = float(series.min())
                ymax = float(series.max())
                if ymax == ymin:
                    margin = 1.0
                else:
                    margin = 0.05 * (ymax - ymin)
                ax.set_ylim(ymin - margin, ymax + margin)

        fig.tight_layout()
        return [line for d in lines.values() for line in d.values()]

    ani = FuncAnimation(
        fig,
        update,
        frames=n_frames,
        init_func=init,
        blit=False,
        interval=100,  # ms (only affects interactive preview)
    )

    writer = PillowWriter(fps=fps)
    ani.save(outfile, writer=writer)
    plt.close(fig)
    print(f"Saved two_weeks-all-sensors animation to {outfile}")


make_two_weeks_all_sensors_animation(
    cleaned_df=cleaned_df,
    outfile="../all_data_files/two_weeks_all_sensors.gif",
    days_back=14,
    fps=10,
    frame_step="30min",   # one frame every 30 minutes
)

Animating last 14 days, 6 devices, 7 parameters, 673 frames...


Saved two_weeks-all-sensors animation to ../all_data_files/two_weeks_all_sensors.gif
